# Candidate Generation Pipeline

## Objective

Generate candidate Global Parent (GP) homologations for manual review.

This notebook **never decides** whether two Global Parents represent the same company.

Its responsibility is to:

1. Normalize GP names.
2. Generate deterministic normalization groups.
3. Generate candidate homologations using independent matching algorithms.
4. Export review files for manual validation.

The reviewed outputs produced by this notebook become the input for **Notebook 2 – GP Dictionary**.

---

## Current Matching Algorithms

- Exact Normalization
- Fuzzy Matching

Future algorithms can be added independently:

- Embedding Similarity
- TF-IDF
- LLM Suggestions

---

## Outputs

```
Output/
│
├── normalization_groups.csv
└── fuzzy_review.csv
```

The normalization groups are deterministic and require no manual review.

The fuzzy review file must be manually reviewed before being used by Notebook 2.

In [1]:
from pathlib import Path
import re
import unicodedata
import pandas as pd
from rapidfuzz import fuzz
from rapidfuzz import process

In [2]:
# ============================================================
# Files
# ============================================================

INPUT_FILE = Path(r'C:\Users\reyiv\Documents\Quaker Houghton\Codigos\limpieza-brandname rep\limpieza-brandname\GP Cleaning\Industrial_Market.csv')
OUTPUT_FOLDER = Path("Output")

OUTPUT_FOLDER.mkdir(exist_ok=True)

# ============================================================
# Catalogue
# ============================================================

GP_COLUMN = "Global Parent"

# ============================================================
# Normalization
# ============================================================

LEGAL_SUFFIXES = [

    'INC'
    "INC.",
    "INCORPORATED",

    "CORP",
    "CORPORATION",

    "COMPANY",
    "CO",

    "LIMITED",
    "LTD",

    "LLC",
    "PLC",
    "LP",
    "GMBH",
    "SA",
    "BV",
    "NV"

]

# ============================================================
# Fuzzy Matching
# ============================================================

FUZZY_SCORER = fuzz.token_sort_ratio
FUZZY_THRESHOLD = 85
MAX_CANDIDATES = 10

In [3]:
class CandidateGenerationPipeline:
    
    """
    Generate candidate Global Parent homologations.

    Workflow
    --------
    Customer Catalogue
            ↓
    Normalize GP names
            ↓
    Generate Normalization Groups
            ↓
    Run Candidate Algorithms
            ↓
    Export Review Files
    """

    def __init__(
        self,
        input_file,
        output_folder,
        gp_column,
        legal_suffixes,
        fuzzy_threshold,
        fuzzy_scorer,
        max_candidates,
        
    ):
        """
        Initialize the pipeline configuration.
        """

        # Configuration
        self.input_file = Path(input_file)
        self.output_folder = Path(output_folder)

        self.gp_column = gp_column

        self.legal_suffixes = legal_suffixes

        self.fuzzy_threshold = fuzzy_threshold
        self.fuzzy_scorer = fuzzy_scorer
        self.max_candidates = max_candidates

        # Internal data
        self.catalogue = None
        self.normalization_groups = None
        self.fuzzy_review = None

        self.normalization_groups_export = None
        self.fuzzy_review_export = None


    def _load_catalogue(self):
        # Load customer catalogue
        self.catalogue = pd.read_csv(self.input_file)
        print(f"Loaded {len(self.catalogue):,} rows.")


    def _validate_catalogue(self):
        # Validate the input catalogue.

        if self.gp_column not in self.catalogue.columns:

                raise ValueError(
                f"Column '{self.gp_column}' was not found."
                )

        self.catalogue = self.catalogue.copy()

        self.catalogue = self.catalogue.dropna(
                subset=[self.gp_column]
        )

        self.catalogue[self.gp_column] = (
                self.catalogue[self.gp_column]
                .astype(str)
                .str.strip()
        )


    # ==========================================================
    # Normalization
    # ==========================================================

    def _remove_accents(self, text):
        """
        Remove accents and diacritics.
        """

        text = unicodedata.normalize("NFKD", text)

        return "".join(
            character
            for character in text
            if not unicodedata.combining(character)
        )


    def _remove_legal_suffixes(self, text):
        """
        Remove common legal suffixes.
        """

        words = text.split()

        words = [
            word
            for word in words
            if word not in self.legal_suffixes
        ]

        return " ".join(words)


    def _normalize_gp(self, gp):
        """
        Normalize a Global Parent name.

        The normalization is deterministic and is only used to
        generate candidate groups. It never modifies the original
        customer catalogue.
        """

        if pd.isna(gp):
            return ""

        gp = str(gp).upper()
        gp = self._remove_accents(gp)
        gp = re.sub(r"[^\w\s]", " ", gp)
        gp = self._remove_legal_suffixes(gp)
        gp = re.sub(r"\s+", " ", gp).strip()

        return gp


    def _create_normalization_groups(self):
        """
        Create deterministic normalization groups from the unique
        Global Parent names.
        """

        groups = (
            self.catalogue[[self.gp_column]]
            .drop_duplicates()
            .copy()
        )

        groups["Normalized GP"] = (
            groups[self.gp_column]
            .apply(self._normalize_gp)
        )

        groups = groups.sort_values(
            by=["Normalized GP", self.gp_column]
        ).reset_index(drop=True)

        self.normalization_groups = groups


    def _print_summary(self):
        """
        Print a summary of the pipeline execution.
        """

        total_rows = len(self.catalogue)

        unique_gp = (
            self.catalogue[self.gp_column]
            .nunique()
        )

        normalized_gp = (
            self.normalization_groups["Normalized GP"]
            .nunique()
        )

        print("\nPipeline Summary")
        print("-" * 40)
        print(f"Catalogue rows      : {total_rows:,}")
        print(f"Unique GPs          : {unique_gp:,}")
        print(f"Normalized GPs      : {normalized_gp:,}")
        print(f"Automatic merges    : {unique_gp - normalized_gp:,}")
        print(f"Fuzzy candidates    : {len(self.fuzzy_review):,}")


    def _get_unique_gps(self):
        """
        Return the unique normalized Global Parents.
        """

        return (
            self.normalization_groups
            [["Normalized GP"]]
            .drop_duplicates()
            .sort_values("Normalized GP")
            .reset_index(drop=True)
        )


    def _run_fuzzy_matching(self):
        """
        Generate fuzzy matching candidates.
        """

        unique_gps = self._get_unique_gps()

        names = unique_gps["Normalized GP"].tolist()

        matches = []

        for gp in names:

            results = process.extract(
                query=gp,
                choices=names,
                scorer=self.fuzzy_scorer,
                limit=self.max_candidates
            )

            for candidate, score, _ in results:

                if candidate == gp:
                    continue

                if score < self.fuzzy_threshold:
                    continue

                matches.append({
                    "GP": gp,
                    "Candidate": candidate,
                    "Score": score
                })

        self.fuzzy_review = pd.DataFrame(matches)


    def _remove_duplicate_pairs(self):
        """
        Remove mirrored fuzzy matches.
        """

        if self.fuzzy_review.empty:
            return

        pairs = self.fuzzy_review.copy()

        pairs["Pair"] = pairs.apply(
            lambda row: tuple(sorted([row["GP"], row["Candidate"]])),
            axis=1
        )

        pairs = (
            pairs
            .sort_values("Score", ascending=False)
            .drop_duplicates("Pair")
            .drop(columns="Pair")
            .reset_index(drop=True)
        )

        self.fuzzy_review = pairs


    def _sort_fuzzy_review(self):
        """
        Sort fuzzy candidates.
        """

        self.fuzzy_review = (
            self.fuzzy_review
            .sort_values(
                by=["Score", "GP"],
                ascending=[False, True]
            )
            .reset_index(drop=True)
        )






    def _export_outputs(self):

        self.output_normalization_groups = (
            self.normalization_groups.copy()
        )

        self.output_fuzzy_review = (
            self.fuzzy_review.copy()
        )

        self.output_fuzzy_review["Decision"] = ""



        self.output_normalization_groups.to_csv(
            self.output_folder / "normalization_pairs.csv",
            index=False
        )
        self.output_fuzzy_review.to_csv(
            self.output_folder / "fuzzy_candidates.csv",
            index=False
        )


    def run(self):

        # Load
        self._load_catalogue()
        self._validate_catalogue()

        # Generate candidates
        self._create_normalization_groups()
        self._run_fuzzy_matching()

        # Clean outputs
        self._remove_duplicate_pairs()
        self._sort_fuzzy_review()

        # Export
        self._export_outputs()

        # Summary
        self._print_summary()


In [4]:
pipeline = CandidateGenerationPipeline(
    input_file=INPUT_FILE,
    output_folder=OUTPUT_FOLDER,
    gp_column=GP_COLUMN,
    legal_suffixes=LEGAL_SUFFIXES,
    fuzzy_threshold=FUZZY_THRESHOLD,
    fuzzy_scorer=FUZZY_SCORER,
    max_candidates=MAX_CANDIDATES,
)

In [5]:
pipeline.run()

normalization_groups = (
    pipeline.output_normalization_groups
)

fuzzy_review = (
    pipeline.output_fuzzy_review
)

Loaded 49,492 rows.

Pipeline Summary
----------------------------------------
Catalogue rows      : 49,483
Unique GPs          : 17,931
Normalized GPs      : 17,931
Automatic merges    : 0
Fuzzy candidates    : 3,496
